In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# set gpu to 1
torch.cuda.set_device(0)

In [2]:
base_model_id = "mistralai/Mistral-7B-v0.1"

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True,
)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="cuda:1",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(base_model_id, add_bos_token=True, trust_remote_code=True)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
torch.__version__

'2.1.1+cu118'

In [4]:
from peft import PeftModel
base_model = PeftModel.from_pretrained(base_model, "../train/mistral-nordavind-8bit-768/checkpoint-1000")

In [6]:
from transformers.generation import GenerationConfig

system_prompt = 'Du er "Nordavind", en hjelpsom assistent.'
def make_prompt(inst, inp, sys=True):
    if sys:
        return f"""<s>{system_prompt} [INST] {inst} {inp} [/INST]"""
    return f"""<s>[INST] {inst} {inp} [/INST]"""

USR_PROMPT = "Gi meg fem oppskrifter med følgende ingredienser"
Q = make_prompt("Hvordan lager man lasagne?", inp="")
Q = make_prompt("Hvilke ingredienser trenger man typisk til taco?", inp="")
print(Q)
model_input = tokenizer(Q, return_tensors="pt").to("cuda:1")
base_model.eval();
with torch.no_grad():
    gen = tokenizer.decode(
        base_model.generate(**model_input, max_new_tokens=200, repetition_penalty=1.15)[0],
        skip_special_tokens=True
    )
    print(gen)


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


<s>Du er "Nordavind", en hjelpsom assistent. [INST] Hvordan lager man lasagne?  [/INST] \n 
Du er "Nordavind", en hjelpsom assistent. [INST] Hvordan lager man lasagne?  [/INST] \n 1. Lag en tomatsaus ved å koke tomatpuree, olivenolje og salt i en stor pot over middels varme for ca. 20
